# Phase 8 - QLoRA Fine-Tuning of Qwen2.5-3B

**This notebook trains the model. Expected time: 45-90 minutes on T4.**

What happens:
1. Loads Qwen2.5-3B base in 4-bit NF4 quantization
2. Applies LoRA (rank=16, alpha=32) on all 7 projection layers
3. Trains for 3 epochs with cosine LR + paged_adamw_8bit optimizer
4. Saves the LoRA adapter to `models/adapters/telecom-ticket-triage/`

**The adapter is ~60 MB (committed to git). The base model (~6 GB) stays on Drive.**

## Prerequisites
- Phase 7 complete (`training/prepared/{train,validation,test}/` exist on Drive)
- **T4 GPU runtime** (required for 4-bit model + LoRA training)

**Run cells in order.**

In [ ]:
# Cell 1 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# Cell 2 - Set REPO_DIR
import os
REPO_DIR = '/content/drive/MyDrive/telecom-support-ticket-triage'
assert os.path.isdir(REPO_DIR), 'REPO_DIR not found: ' + REPO_DIR
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Cell 3 - Git pull
!git pull
!git log --oneline -5

In [ ]:
# Cell 4 - Install dependencies
!pip install -q -r training/requirements-colab.txt
!pip install -q triton==2.3.0
print('Done.')

In [ ]:
# Cell 5 - GPU check (MUST have GPU for training)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('NO GPU DETECTED. Go to Runtime -> Change runtime type -> T4 GPU, then re-run all cells.')
print('GPU: ', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 6 - Verify Phase 7 prepared datasets exist
from pathlib import Path
import json
prepared_dir = Path('training/prepared')
meta_file = prepared_dir / 'meta.json'
if meta_file.exists():
    meta = json.loads(meta_file.read_text())
    print('Phase 7 metadata:')
    print(json.dumps(meta, indent=2))
all_ok = True
for name in ['train', 'validation', 'test']:
    path = prepared_dir / name
    if not path.exists():
        print('MISSING:', path)
        all_ok = False
if not all_ok:
    raise RuntimeError('Prepared datasets missing. Run Phase 7 notebook first.')
print('All prepared datasets found. Ready to train.')

In [ ]:
# Cell 7 - RUN TRAINING
# train.py: loads base model -> applies LoRA -> trains 3 epochs -> saves adapter
# Estimated runtime on T4: 45-90 minutes
# Watch for: loss decreasing each epoch, eval_loss lower than train_loss
# Best checkpoint (lowest eval_loss) is automatically reloaded at the end
!python training/train.py \
    --model-path models/base/Qwen2.5-3B \
    --max-length 512 \
    --epochs 3 \
    --batch-size 2 \
    --grad-accum 8 \
    --learning-rate 2e-4 \
    --lora-rank 16 \
    --lora-alpha 32 \
    --output-dir models/adapters/telecom-ticket-triage
# NOTE: If Cell 10 in Phase 7 showed >5% truncation, add: --max-length 768

In [ ]:
# Cell 8 - Review training metrics
import json
from pathlib import Path
log_path = Path('training/training_log.json')
if not log_path.exists():
    print('training_log.json not found - check if Cell 7 completed successfully.')
else:
    log = json.loads(log_path.read_text())
    print('=== Training metrics ===')
    for k, v in log.get('train_metrics', {}).items():
        val = round(v, 4) if isinstance(v, float) else v
        print(' ', k, ':', val)
    print()
    print('=== Final eval metrics (best checkpoint) ===')
    for k, v in log.get('eval_metrics', {}).items():
        val = round(v, 4) if isinstance(v, float) else v
        print(' ', k, ':', val)

In [ ]:
# Cell 9 - Verify adapter files were saved
from pathlib import Path
adapter_dir = Path('models/adapters/telecom-ticket-triage')
if not adapter_dir.exists():
    print('ERROR: adapter dir not found. Check Cell 7 output for errors.')
else:
    files = list(adapter_dir.iterdir())
    print('Adapter directory:', len(files), 'files')
    for f in sorted(files):
        print(' ', f.name, '-', round(f.stat().st_size / 1024, 1), 'KB')
    meta_f = adapter_dir / 'triage_adapter_meta.json'
    if meta_f.exists():
        import json
        print('\nAdapter meta:', json.loads(meta_f.read_text()))

In [ ]:
# Cell 10 - Inference smoke test
# Loads base model + LoRA adapter and runs one test ticket.
# MUST produce valid JSON {category, priority, department} before proceeding to Phase 9.
import torch, json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_PATH    = 'models/base/Qwen2.5-3B'
ADAPTER_PATH = 'models/adapters/telecom-ticket-triage'

SYSTEM = (
    'You are a support-ticket triage classifier for a telecom company. '
    'Given a customer support ticket, respond with ONLY a strict JSON object '
    'with exactly these keys: "category", "priority", "department". '
    'No explanation, no extra text, no markdown fences.\n\n'
    'category must be one of: Billing, Technical, Account, Refund, General\n'
    'priority must be one of: Critical, High, Medium, Low\n'
    'department must be one of: Finance, Technical, Account, Refunds, General Support'
)
TICKET = 'My mobile data stopped working completely since yesterday morning. Tried restarting and reinserting SIM. Nothing helped. Need this fixed urgently as I use data for work.'

print('Loading tokenizer...')
tok = AutoTokenizer.from_pretrained(BASE_PATH)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

print('Loading base model in 4-bit...')
base = AutoModelForCausalLM.from_pretrained(BASE_PATH, quantization_config=bnb,
    device_map='auto', trust_remote_code=True)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()

msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': TICKET}]
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors='pt').to(model.device)

print('Running inference...')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
        pad_token_id=tok.eos_token_id)

generated = out[0][inputs['input_ids'].shape[1]:]
response  = tok.decode(generated, skip_special_tokens=True).strip()
print('\nModel output:', response)

try:
    p = json.loads(response)
    cats  = {'Billing','Technical','Account','Refund','General'}
    pris  = {'Critical','High','Medium','Low'}
    depts = {'Finance','Technical','Account','Refunds','General Support'}
    assert p.get('category') in cats, 'Bad category: ' + str(p.get('category'))
    assert p.get('priority') in pris, 'Bad priority: ' + str(p.get('priority'))
    assert p.get('department') in depts, 'Bad dept: ' + str(p.get('department'))
    print('\nInference test PASSED:')
    print('  category:  ', p['category'])
    print('  priority:  ', p['priority'])
    print('  department:', p['department'])
except Exception as e:
    print('\nInference test FAILED:', e)
    print('Model may need more training or output format needs review.')

In [ ]:
# Cell 11 - Git commit adapter metadata + training log
# adapter_model.safetensors is excluded from git by .gitignore (*safetensors)
# We commit: triage_adapter_meta.json, adapter_config.json, tokenizer files, training_log.json
!git add models/adapters/telecom-ticket-triage/triage_adapter_meta.json
!git add models/adapters/telecom-ticket-triage/adapter_config.json
!git add models/adapters/telecom-ticket-triage/tokenizer.json
!git add models/adapters/telecom-ticket-triage/tokenizer_config.json
!git add models/adapters/telecom-ticket-triage/special_tokens_map.json
!git add training/training_log.json
!git status
!git commit -m 'Phase 8 complete: QLoRA adapter trained and saved'
!git push
print('\nPHASE 8 COMPLETE')
print('The LoRA adapter weights (adapter_model.safetensors) stay on Drive only.')
print('Next: run Phase 9 (evaluate.py) to measure test set accuracy and F1.')